# 00 — Environment test

**Environment checks only.** Run this first from a fresh kernel after creating the pinned virtual environment. It confirms that this clone can import the required data-science and GIS stack, resolves the repository root portably, and performs only in-memory smoke tests. It does not download data, create derived project outputs, or run the analytical pipeline.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_support import resolve_project_root
PROJECT_ROOT = resolve_project_root(PROJECT_ROOT)
print('Repository root:', PROJECT_ROOT)
print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])

## Local-input preflight

This calls the same read-only source-presence check as `scripts/run_project.py --mode preflight`. It does not open, download, alter, or validate the contents of raw files.

In [ ]:
import pandas as pd
from src.project_run import raw_data_preflight

preflight = raw_data_preflight()
display(pd.DataFrame(preflight['groups']))
print('Raw-input preflight status:', preflight['status'])

## Pinned package contract

The table compares the installed package versions with the exact pins in `requirements.txt`. A mismatch should be corrected by rebuilding the local virtual environment before running the reproducible pipeline.

In [ ]:
import importlib.metadata as metadata
from src.notebook_support import pinned_requirements

required = pinned_requirements(PROJECT_ROOT)
package_versions = pd.DataFrame([
    {'package': package, 'required': expected, 'installed': metadata.version(package), 'matches': metadata.version(package) == expected}
    for package, expected in required.items()
])
assert package_versions['matches'].all(), 'At least one installed package differs from requirements.txt.'
display(package_versions)

## Geospatial and machine-learning smoke tests

These small synthetic examples confirm CRS transformation and numerical preprocessing only. They do not use project data or write figures.

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler
from src.config import SPATIAL

sample = gpd.GeoDataFrame(
    {'location': ['Lisbon test point', 'Porto test point']},
    geometry=[Point(-9.1393, 38.7223), Point(-8.6291, 41.1579)],
    crs='EPSG:4326',
)
projected = sample.to_crs(SPATIAL.analysis_crs)
assert projected.crs.to_string() == SPATIAL.analysis_crs == 'EPSG:3763'

synthetic_features = pd.DataFrame({'forest_shrub_share_2km': [0.10, 0.25, 0.70, 0.85], 'fire_years_previous_10y_2km': [0, 1, 3, 5]})
scaled = StandardScaler().fit_transform(synthetic_features)
assert np.isfinite(scaled).all()
display(projected)
print('EPSG:3763 transformation and in-memory numerical-preprocessing checks passed.')

## Next step

Run `python scripts/run_project.py --mode preflight` in a terminal to check that local raw inputs are present without downloading or changing them. Then continue with `01_data_collection.ipynb`.